In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion10-series-tiempo-inferencia")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

In [9]:
from pyspark.sql import functions as F
import os

RUTA_GOLD = "./artifacts/campo_electrico_particionado"
os.makedirs("./artifacts", exist_ok=True)

# 1. Leer los 3 CSVs
df_ce = spark.read.option("header", "true").option("inferSchema", "true").csv("data/campo_electrico.csv")
df_cm = spark.read.option("header", "true").option("inferSchema", "true").csv("data/campo_magnetico.csv")
df_amb = spark.read.option("header", "true").option("inferSchema", "true").csv("data/variables_ambientales.csv")

# 2. Normalizar nombres de columnas (remover puntos y espacios residuales)
for col_name in df_amb.columns:
    col_limpia = col_name.strip().replace(".", "")
    if col_limpia != col_name:
        df_amb = df_amb.withColumnRenamed(col_name, col_limpia)

for col_name in df_ce.columns:
    col_limpia = col_name.strip().replace(".", "")
    if col_limpia != col_name:
        df_ce = df_ce.withColumnRenamed(col_name, col_limpia)

for col_name in df_cm.columns:
    col_limpia = col_name.strip().replace(".", "")
    if col_limpia != col_name:
        df_cm = df_cm.withColumnRenamed(col_name, col_limpia)

# 3. Columnas obligatorias de la guía S04/S10
VARIABLES_8 = ["CE", "CM", "TempOut", "OutHum", "WindSpeed", "Bar", "SolarRad", "UVIndex"]

# 4. Integrar, filtrar error de sensor (-999999) y proyectar
df_gold = (
    df_ce.join(df_cm, on="DateTime", how="inner")
         .join(df_amb, on="DateTime", how="inner")
         .select("DateTime", *VARIABLES_8)
         .filter(F.col("CM") != -999999)
         .dropDuplicates(["DateTime"])
         .na.drop()
)

# 5. Guardar en Parquet particionado
df_gold.write.mode("overwrite").parquet(RUTA_GOLD)
df_gold = spark.read.parquet(RUTA_GOLD)

print(f"Total filas Gold: {df_gold.count():,}")
df_gold.orderBy("DateTime").show(5)

Total filas Gold: 93,027
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+
|           DateTime|   CE|     CM|TempOut|OutHum|WindSpeed|  Bar|SolarRad|UVIndex|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+
|2026-05-10 00:00:00|-0.28|24194.3|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|
|2026-05-10 00:01:00|-0.34|24193.8|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|
|2026-05-10 00:02:00|-0.39|24193.6|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|
|2026-05-10 00:03:00|-0.43|24194.4|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|
|2026-05-10 00:04:00|-0.38|24195.2|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+
only showing top 5 rows


In [12]:
from pyspark.sql.window import Window

VARIABLES_8 = ["CE", "CM", "TempOut", "OutHum", "WindSpeed", "Bar", "SolarRad", "UVIndex"]
orden_temporal = Window.orderBy("DateTime")

# Construir label en t+1 con lead()
dataset_ml = (
    df_gold
    .withColumn("label", F.lead("CE", 1).over(orden_temporal))
    .select("DateTime", *VARIABLES_8, "label")
    .na.drop()
)

# División cronológica 80/20 por fila (no aleatoria)
numerado = dataset_ml.withColumn("numeroFila", F.row_number().over(orden_temporal))
corte = int(numerado.count() * 0.80)

train = numerado.filter(F.col("numeroFila") <= corte).drop("numeroFila")
test = numerado.filter(F.col("numeroFila") > corte).drop("numeroFila")

fecha_max_train = train.agg(F.max("DateTime")).first()[0]
fecha_min_test = test.agg(F.min("DateTime")).first()[0]

print(f"Filas dataset supervisado: {dataset_ml.count():,}")
print(f"Entrenamiento: {train.count():,} filas (hasta {fecha_max_train})")
print(f"Prueba: {test.count():,} filas (desde {fecha_min_test})")
dataset_ml.show(5)

26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

Filas dataset supervisado: 93,026
Entrenamiento: 74,420 filas (hasta 2026-08-19 22:39:00)


26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

Prueba: 18,606 filas (desde 2026-08-19 22:40:00)
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+
|           DateTime|   CE|     CM|TempOut|OutHum|WindSpeed|  Bar|SolarRad|UVIndex|label|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+
|2026-05-10 00:00:00|-0.28|24194.3|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.34|
|2026-05-10 00:01:00|-0.34|24193.8|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.39|
|2026-05-10 00:02:00|-0.39|24193.6|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.43|
|2026-05-10 00:03:00|-0.43|24194.4|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.38|
|2026-05-10 00:04:00|-0.38|24195.2|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.35|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+
only showing top 5 rows


26/10/23 03:21:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [14]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluador_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluador_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluador_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

def evaluar(predicciones, nombre):
    resultado = {
        "Modelo": nombre,
        "RMSE": evaluador_rmse.evaluate(predicciones),
        "MAE": evaluador_mae.evaluate(predicciones),
        "R2": evaluador_r2.evaluate(predicciones),
    }
    print(f"{nombre}: RMSE={resultado['RMSE']:.4f}  MAE={resultado['MAE']:.4f}  R2={resultado['R2']:.4f}")
    return resultado

test_persistencia = test.withColumn("prediction", F.col("CE"))
resultado_persistencia = evaluar(test_persistencia, "Persistencia (CE_t+1 = CE_t)")

26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

Persistencia (CE_t+1 = CE_t): RMSE=0.2085  MAE=0.1314  R2=0.9464


26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:21:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

In [15]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# 1. Ensamblar las 8 variables en el vector 'features'
ensamblador = VectorAssembler(inputCols=VARIABLES_8, outputCol="features")
train_ml = ensamblador.transform(train)
test_ml = ensamblador.transform(test)

# 2. Entrenar regresión lineal
lr = LinearRegression(featuresCol="features", labelCol="label")
modelo_lr = lr.fit(train_ml)

print("Coeficientes:", modelo_lr.coefficients)
print("Intercepto:", modelo_lr.intercept)

predicciones_lr = modelo_lr.transform(test_ml)
resultado_lr = evaluar(predicciones_lr, "LinearRegression")

26/10/23 03:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

Coeficientes: [0.9795627182811428,-8.198094865649054e-08,0.0021777177554351207,3.815462278265169e-05,-0.0018841725224271993,0.0010813912892555855,-8.34019716666204e-07,0.0015475001142227515]
Intercepto: -1.0834841727121014


26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

LinearRegression: RMSE=0.2070  MAE=0.1320  R2=0.9472


26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

In [16]:
from pyspark.ml.regression import RandomForestRegressor

# Entrenar Random Forest con 50 árboles
rf = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(train_ml)

predicciones_rf = modelo_rf.transform(test_ml)
resultado_rf = evaluar(predicciones_rf, "RandomForestRegressor")

# Mostrar importancia de cada variable
importancias = sorted(zip(VARIABLES_8, modelo_rf.featureImportances.toArray()), key=lambda par: -par[1])
print("\nImportancia de características:")
for variable, importancia in importancias:
    print(f"{variable:10s} {importancia:.4f}")

26/10/23 03:24:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:24:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

RandomForestRegressor: RMSE=0.2472  MAE=0.1665  R2=0.9247

Importancia de características:
CE         0.8753
TempOut    0.0303
CM         0.0301
OutHum     0.0265
WindSpeed  0.0156
UVIndex    0.0088
Bar        0.0073
SolarRad   0.0061


In [17]:
import pandas as pd

comparacion = pd.DataFrame([resultado_persistencia, resultado_lr, resultado_rf])
comparacion["MejoraRMSE_vs_persistencia_%"] = (
    100 * (resultado_persistencia["RMSE"] - comparacion["RMSE"]) / resultado_persistencia["RMSE"]
)
comparacion

,Modelo,RMSE,MAE,R2,MejoraRMSE_vs_persistencia_%
0,Persistencia (CE_t+1 = CE_t),0.208500,0.131396,0.946419,0.000000
1,LinearRegression,0.207048,0.131966,0.947163,0.696401
2,RandomForestRegressor,0.247173,0.166454,0.924700,-18.547762


In [18]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

RUTA_MODELO = "./artifacts/models/modelo_ce_lr"

# 1. Definir y entrenar el Pipeline completo (VectorAssembler + LinearRegression)
pipeline_ganador = Pipeline(stages=[
    VectorAssembler(inputCols=VARIABLES_8, outputCol="features"),
    LinearRegression(featuresCol="features", labelCol="label")
])

modelo_final = pipeline_ganador.fit(train)

# 2. Guardar en disco sobreescribiendo si ya existe
modelo_final.write().overwrite().save(RUTA_MODELO)

# 3. Recargar el artefacto guardado para verificar persistencia
modelo_cargado = PipelineModel.load(RUTA_MODELO)
print("PipelineModel guardado y recargado exitosamente en:", RUTA_MODELO)

# 4. Probar inferencia sobre 5 filas del conjunto de prueba
predicciones_muestra = modelo_cargado.transform(test).select("DateTime", *VARIABLES_8, "label", "prediction")
predicciones_muestra.show(5, truncate=False)

26/10/23 03:26:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

PipelineModel guardado y recargado exitosamente en: ./artifacts/models/modelo_ce_lr
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+---------------------+
|DateTime           |CE   |CM     |TempOut|OutHum|WindSpeed|Bar  |SolarRad|UVIndex|label|prediction           |
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+---------------------+
|2026-08-19 22:40:00|0.03 |24257.2|13.4   |90.0  |0.0      |951.0|0.0     |0.0    |-0.02|0.0049325306239138555|
|2026-08-19 22:41:00|-0.02|24256.7|13.4   |90.0  |0.0      |951.0|0.0     |0.0    |-0.07|-0.044045564299669104|
|2026-08-19 22:42:00|-0.07|24256.0|13.4   |90.0  |0.0      |951.0|0.0     |0.0    |-0.1 |-0.09302364282706221 |
|2026-08-19 22:43:00|-0.1 |24257.3|13.4   |90.0  |0.0      |951.0|0.0     |0.0    |-0.09|-0.12241063095072979 |
|2026-08-19 22:44:00|-0.09|24257.5|13.4   |90.0  |0.0      |951.0|0.0     |0.0    |-0.04|-0.11261502016410807 |
+-------------------

26/10/23 03:26:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 03:26:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

In [19]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
from pyspark.sql import functions as F
import os, shutil

# 1. Esquema del evento que emite el productor
esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("estacionId", StringType())
    .add("CE", DoubleType())
    .add("CM", DoubleType())
    .add("TempOut", DoubleType())
    .add("OutHum", DoubleType())
    .add("WindSpeed", DoubleType())
    .add("Bar", DoubleType())
    .add("SolarRad", DoubleType())
    .add("UVIndex", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

# 2. Leer stream desde Kafka
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "campo-electrico-eventos")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

eventos = (
    crudo.select(F.col("value").cast("string").alias("json_str"))
    .select(F.from_json(F.col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
)

# Aplicar el PipelineModel cargado (inferencia sin estado / stateless)
predicciones_stream = modelo_cargado.transform(eventos)

# 3. Rutas de persistencia
RUTA_PREDICCIONES_STREAM = "./artifacts/predicciones_stream"
CHECKPOINT_STREAM = "./artifacts/chk_predicciones_stream"

# Limpieza preventiva de checkpoints anteriores
shutil.rmtree(CHECKPOINT_STREAM, ignore_errors=True)
shutil.rmtree(RUTA_PREDICCIONES_STREAM, ignore_errors=True)

# 4. Función de persistencia por micro-lote
def guardar_predicciones_microlote(df_microlote, id_lote):
    filas = df_microlote.count()
    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas")
        return
    df_microlote.select(
        F.lit(id_lote).alias("batchId"),
        "estacionId",
        (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora"),
        "CE",
        F.col("prediction").alias("prediccionSiguienteCE")
    ).write.mode("append").parquet(RUTA_PREDICCIONES_STREAM)
    print(f"Micro-lote {id_lote}: {filas} predicciones generadas y guardadas.")

# Arrancar consulta streaming
consulta_predicciones = (
    predicciones_stream.writeStream
    .foreachBatch(guardar_predicciones_microlote)
    .option("checkpointLocation", CHECKPOINT_STREAM)
    .start()
)

26/10/23 03:27:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Micro-lote 0: sin filas


Micro-lote 1: 1 predicciones generadas y guardadas.
Micro-lote 2: 1 predicciones generadas y guardadas.
Micro-lote 3: 1 predicciones generadas y guardadas.
Micro-lote 4: 1 predicciones generadas y guardadas.
Micro-lote 5: 1 predicciones generadas y guardadas.
Micro-lote 6: 1 predicciones generadas y guardadas.
Micro-lote 7: 1 predicciones generadas y guardadas.
Micro-lote 8: 1 predicciones generadas y guardadas.
Micro-lote 9: 1 predicciones generadas y guardadas.
Micro-lote 10: 1 predicciones generadas y guardadas.
Micro-lote 11: 1 predicciones generadas y guardadas.
Micro-lote 12: 1 predicciones generadas y guardadas.
Micro-lote 13: 1 predicciones generadas y guardadas.
Micro-lote 14: 1 predicciones generadas y guardadas.
Micro-lote 15: 1 predicciones generadas y guardadas.
Micro-lote 16: 1 predicciones generadas y guardadas.
Micro-lote 17: 1 predicciones generadas y guardadas.
Micro-lote 18: 1 predicciones generadas y guardadas.
Micro-lote 19: 1 predicciones generadas y guardadas.
Mi

In [20]:
# Detener el stream
consulta_predicciones.stop()

# Mostrar las predicciones en tiempo real guardadas
spark.read.parquet(RUTA_PREDICCIONES_STREAM).orderBy(F.desc("batchId")).show(10, truncate=False)

26/10/23 03:29:41 ERROR FileFormatWriter: Aborting job 6ede87a2-411c-4f92-9063-c30b22c0da05.
java.lang.InterruptedException
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.acquireSharedInterruptibly(AbstractQueuedSynchronizer.java:1134)
	at scala.concurrent.impl.Promise$DefaultPromise.tryAwait0(Promise.scala:253)
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:265)
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:104)
	at org.apache.spark.util.ThreadUtils$.awaitReady(ThreadUtils.scala:417)
	at org.apache.spark.scheduler.DAGScheduler.runJob(DAGScheduler.scala:1052)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2496)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeWrite$4(FileFormatWriter.scala:318)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.writeAndCommit(FileFormatWriter.scala:284)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeWrite

+-------+--------------+-----------------------+----+---------------------+
|batchId|estacionId    |fechaHora              |CE  |prediccionSiguienteCE|
+-------+--------------+-----------------------+----+---------------------+
|37     |estacion-ns-01|2026-10-23 03:29:37.256|1.72|1.644118109563312    |
|36     |estacion-ns-01|2026-10-23 03:29:34.253|1.66|1.5849882178508778   |
|35     |estacion-ns-01|2026-10-23 03:29:31.248|1.7 |1.6239249617320701   |
|34     |estacion-ns-01|2026-10-23 03:29:28.245|1.65|1.5747925983815398   |
|33     |estacion-ns-01|2026-10-23 03:29:25.242|1.6 |1.5254337391532653   |
|32     |estacion-ns-01|2026-10-23 03:29:22.239|1.62|1.54517227599321     |
|31     |estacion-ns-01|2026-10-23 03:29:19.237|1.68|1.60470382879689     |
|30     |estacion-ns-01|2026-10-23 03:29:16.233|1.63|1.5553924521651505   |
|29     |estacion-ns-01|2026-10-23 03:29:13.231|1.61|1.5354388790716471   |
|28     |estacion-ns-01|2026-10-23 03:29:09.431|1.65|1.5747458801967054   |
+-------+---

26/10/23 03:29:41 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 236 cancelled Query [id = 1dd67299-66aa-4db8-89b7-6d27f37fb8cb, runId = 606efbc0-7395-4a20-9ae8-5ecdceae11ba] was stopped SQLSTATE: XXKDA
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:342)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:610)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactor